# NHL Trap Games

Definition: A trap game is about a good team underperforming when they're expected to win

In [1]:
import sys
import os
import polars as pl
from tqdm import tqdm

base_dir = '../'
sys.path.insert(1, base_dir)

import src.config as cfg
import src.utils as utils
import src.sportsipy_utils as sportsipy_utils

## Pull Schedule for Each NHL Team

Note: We are using *5 v 5* results to focus on a team's true underlying ability

In [2]:
# Warning: NHL Schedules not loading - https://github.com/davidjkrause/sportsipy/issues/14
# sportsipy_utils.pull_nhl_schedule('TOR', 2024)

In [57]:
# https://www.naturalstattrick.com/games.php?fromseason=20242025&thruseason=20242025&stype=2&sit=5v5&loc=B&team=All&rate=n
schedule_folder = f'{base_dir}data/natural_stat_trick'

df_games = []
for filename in tqdm(os.listdir(schedule_folder)):
    if 'games_' in filename:
        schedule_filepath = os.path.join(schedule_folder, filename)
        df_games_filepath = pl.scan_csv(schedule_filepath, null_values=["-"]).collect()
        if not df_games_filepath.is_empty():
            df_games += [df_games_filepath]
    
df_games = (pl.concat(df_games)
            .unique(subset=['Game', 'Team'], keep='first')
            .with_columns(
                pl.col('Game').str.split(" - ").list.get(0).str.strip_chars().str.strptime(pl.Datetime,'%Y-%m-%d').alias('Date')
            )
            .select(['Game', 'Team', 'Date', 'GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%' ,'PDO'])
           )
utils.logger.info(f"Loaded {df_games.shape[0]/2} unique games from natural stattrick") 
df_games.head(5)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 92.41it/s]
2025-09-13 11:34:07,815 [1402038357.py:19] [INFO] Loaded 21528.0 unique games from natural stattrick


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64
""" 2021-01-24 - Avalanche 1, Duc…","""Anaheim Ducks""",2021-01-24 00:00:00,2,1,66.67,1.39,2.05,40.42,18.18,96.15,1.143
""" 2016-12-29 - Rangers 6, Coyot…","""New York Rangers""",2016-12-29 00:00:00,1,2,33.33,1.79,1.34,57.29,5.26,86.67,0.919
""" 2010-12-31 - Senators 3, Blue…","""Ottawa Senators""",2010-12-31 00:00:00,2,3,40.0,1.72,1.33,56.46,7.14,76.92,0.841
""" 2017-02-18 - Blues 2, Sabres …","""Buffalo Sabres""",2017-02-18 00:00:00,2,0,100.0,0.74,1.8,28.98,12.5,100.0,1.125
""" 2019-03-11 - Senators 2, Flye…","""Ottawa Senators""",2019-03-11 00:00:00,2,3,40.0,1.35,3.02,30.82,10.0,89.66,0.997


In [82]:
# Warning: for 1000 pairs of (game, team), there were no 5v5 goals scored extracted GF/GA numbers 
df_missing = df_games.filter(pl.col("GF%").is_null())
utils.logger.info(f"Found {df_missing.shape[0]} / {df_games.shape[0]} rows without any 5v5 goals")

2025-09-13 11:49:00,934 [3340401753.py:3] [INFO] Found 1050 / 43056 rows without any 5v5 goals


## Identify Trap Games

In [127]:
strength_col = "5v5 Points"
strong_threshold = 1.1
weak_threshold = 0.9

df_games_processed = (
    df_games.filter(pl.col("GF%").is_not_null())
      .unique(subset=['Game', 'Team'], keep='first')
      .sort("Game", descending=False)
      .with_columns([
        pl.when(pl.col("Date").dt.month() >= 7).then(pl.col("Date").dt.year() + 1)
            .otherwise(pl.col("Date").dt.year())
            .alias("Season"),
          pl.when(pl.col("GF%")>50).then(pl.lit(2))
              .when(pl.col("GF%")<50).then(pl.lit(0))
              .otherwise(pl.lit(1))
          .alias("5v5 Points")
      ])
    .with_columns([
        pl.col(strength_col).cum_sum().over(["Team", "Season"]).alias("Season to Date Total incl Current"),
        pl.col("GF%").cum_count().over(["Team", "Season"]).alias("# Games Season to Date incl Current")
    ])
    .with_columns(
        ((pl.col("Season to Date Total incl Current") - pl.col(strength_col))/
         (pl.col("# Games Season to Date incl Current") - 1)).alias(f"Season to Date Average {strength_col}")
    )
    .with_columns([
        ((pl.col(f"Season to Date Average {strength_col}") > strong_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_strong"),
        ((pl.col(f"Season to Date Average {strength_col}") < weak_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_weak")
    ])
)  
                     
display(df_games_processed.tail(5))
df_games_processed.mean()

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool
""" 2025-04-17 - Islanders 1, Blu…","""Columbus Blue Jackets""",2025-04-17 00:00:00,6,1,85.71,2.9,2.92,49.86,23.08,97.14,1.202,2025,2,78,78,0.987013,false,false
""" 2025-04-17 - Lightning 0, Ran…","""New York Rangers""",2025-04-17 00:00:00,3,0,100.0,1.89,2.17,46.51,15.0,100.0,1.15,2025,2,82,81,1.0,false,false
""" 2025-04-17 - Lightning 0, Ran…","""Tampa Bay Lightning""",2025-04-17 00:00:00,0,3,0.0,2.17,1.89,53.49,0.0,85.0,0.85,2025,0,94,81,1.175,true,false
""" 2025-04-17 - Red Wings 3, Map…","""Detroit Red Wings""",2025-04-17 00:00:00,2,2,50.0,2.25,1.93,53.84,6.9,87.5,0.944,2025,1,78,79,0.987179,false,false
""" 2025-04-17 - Red Wings 3, Map…","""Toronto Maple Leafs""",2025-04-17 00:00:00,2,2,50.0,1.93,2.25,46.16,12.5,93.1,1.056,2025,1,88,77,1.144737,true,false


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak
str,str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,2016-09-02 12:12:30.749892,1.927296,1.927272,50.000793,1.863623,1.863614,50.000135,8.501321,91.498861,1.000008,2016.636957,1.0,39.156192,39.141266,NaN,0.200495,0.200519


In [128]:
# Double check season derivation is reasonable given lockout in the 2012-2013 season and pandemic in 2019-2020/2020-2021 seasons
df_games_processed.group_by("Season").agg(pl.col("Game").count()).sort("Game").head(5)

Season,Game
i32,u32
2013,1404
2021,1698
2020,2126
2008,2352
2009,2358


In [129]:
df_games_processed.filter(pl.col("Team")=="Toronto Maple Leafs").head(25)

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool
""" 2007-10-03 - Senators 4, Mapl…","""Toronto Maple Leafs""",2007-10-03 00:00:00,3,3,50.0,1.77,1.66,51.62,15.0,85.71,1.007,2008,1,1,1,NaN,false,false
""" 2007-10-04 - Maple Leafs 2, S…","""Toronto Maple Leafs""",2007-10-04 00:00:00,1,1,50.0,1.94,1.43,57.55,3.85,93.75,0.976,2008,1,2,2,1.0,false,false
""" 2007-10-06 - Canadiens 3, Map…","""Toronto Maple Leafs""",2007-10-06 00:00:00,3,2,60.0,1.6,1.43,52.85,12.0,88.24,1.002,2008,2,4,3,1.0,false,false
""" 2007-10-09 - Hurricanes 7, Ma…","""Toronto Maple Leafs""",2007-10-09 00:00:00,0,3,0.0,1.34,2.6,33.92,0.0,85.71,0.857,2008,0,4,4,1.333333,false,false
""" 2007-10-11 - Islanders 1, Map…","""Toronto Maple Leafs""",2007-10-11 00:00:00,3,0,100.0,2.67,0.92,74.35,11.11,100.0,1.111,2008,2,6,5,1.0,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
""" 2007-11-17 - Senators 0, Mapl…","""Toronto Maple Leafs""",2007-11-17 00:00:00,1,0,100.0,1.68,2.12,44.23,7.69,100.0,1.077,2008,2,24,21,1.1,false,false
""" 2007-11-20 - Bruins 4, Maple …","""Toronto Maple Leafs""",2007-11-20 00:00:00,1,2,33.33,1.56,1.84,45.79,4.35,90.48,0.948,2008,0,24,22,1.142857,true,false
""" 2007-11-23 - Maple Leafs 1, S…","""Toronto Maple Leafs""",2007-11-23 00:00:00,1,2,33.33,1.2,1.7,41.37,7.14,90.48,0.976,2008,0,24,23,1.090909,false,false


In [139]:
df_opponent = df_games_processed.select([
    pl.col("Game"),
    pl.col("Team").alias("Opponent"),
    pl.col(f"Season to Date Average {strength_col}").alias(f"Opponent Season to Date Average {strength_col}"),
    pl.col("is_strong").alias("Opponent is_strong"),
    pl.col("is_weak").alias("Opponent is_weak")
])

df_games_with_opponent = (
    df_games_processed.join(df_opponent, on="Game", how="left")
    .filter(pl.col("Team") != pl.col("Opponent"))
    .with_columns(
        (pl.col(f"Season to Date Average {strength_col}") - pl.col(f"Opponent Season to Date Average {strength_col}")).alias("Delta {strength_col}")
    )
)

assert df_games_with_opponent.shape[0] == df_games_processed.shape[0]

df_games_with_opponent.filter(pl.col(f"Delta {strength_col}").abs() >= 0.2)

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak,Opponent,Opponent Season to Date Average 5v5 Points,Opponent is_strong,Opponent is_weak,Delta {strength_col}
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,f64,bool,bool,f64
""" 2007-09-30 - Kings 1, Ducks 4""","""Los Angeles Kings""",2007-09-30 00:00:00,1,2,33.33,1.19,1.58,42.89,5.0,89.47,0.945,2008,0,0,1,NaN,false,false,"""Anaheim Ducks""",NaN,false,false,NaN
""" 2007-09-30 - Kings 1, Ducks 4""","""Anaheim Ducks""",2007-09-30 00:00:00,2,1,66.67,1.58,1.19,57.11,10.53,95.0,1.055,2008,2,2,1,NaN,false,false,"""Los Angeles Kings""",NaN,false,false,NaN
""" 2007-10-03 - Canadiens 3, Hur…","""Montreal Canadiens""",2007-10-03 00:00:00,0,1,0.0,2.25,2.2,50.52,0.0,95.45,0.955,2008,0,0,1,NaN,false,false,"""Carolina Hurricanes""",NaN,false,false,NaN
""" 2007-10-03 - Canadiens 3, Hur…","""Carolina Hurricanes""",2007-10-03 00:00:00,1,0,100.0,2.2,2.25,49.48,4.55,100.0,1.045,2008,2,2,1,NaN,false,false,"""Montreal Canadiens""",NaN,false,false,NaN
""" 2007-10-03 - Ducks 2, Red Win…","""Detroit Red Wings""",2007-10-03 00:00:00,0,1,0.0,2.01,0.86,70.05,0.0,85.71,0.857,2008,0,0,1,NaN,false,false,"""Anaheim Ducks""",2.0,false,false,NaN
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
""" 2025-04-17 - Capitals 2, Peng…","""Pittsburgh Penguins""",2025-04-17 00:00:00,3,1,75.0,2.28,1.89,54.61,9.09,94.44,1.035,2025,2,69,81,0.8375,false,true,"""Washington Capitals""",1.2125,true,false,-0.375
""" 2025-04-17 - Flames 5, Kings …","""Los Angeles Kings""",2025-04-17 00:00:00,0,5,0.0,1.95,2.4,44.78,0.0,82.14,0.821,2025,0,103,78,1.337662,true,false,"""Calgary Flames""",0.923077,false,false,0.414585
""" 2025-04-17 - Flames 5, Kings …","""Calgary Flames""",2025-04-17 00:00:00,5,0,100.0,2.4,1.95,55.22,17.86,100.0,1.179,2025,2,74,79,0.923077,false,false,"""Los Angeles Kings""",1.337662,true,false,-0.414585
